# Example: Fixed-Context GG-PA in the Coupled Double-Well

## 1. System Setup

We consider a one-dimensional double-well coordinate `x` coupled to an explicit environment variable `u`. The target coupled equilibrium is

$$
p(x, u) \propto p_0(x)\, e^{-\beta W(x,u)},
$$

where the bare prior is represented by a pretrained diffusion model,

$$
p_0(x) \propto e^{-\beta V_0(x)},
\qquad
V_0(x) = a x^4 - b x^2 + c x,
$$

and the coupling term is

$$
W(x,u) = \frac12 k_b (u-u_{\rm eq})^2 + \frac12 k_c (x-u)^2.
$$

In the GG-PA signal we use

$$
s = (y, u),
$$

where `y` is the diffusion-noised version of `x`, and `u` is the environment coordinate.

## 2. Annealed Context and the Match Condition

For the VP forward kernel

$$
q_t(y \mid x) = \mathcal{N}(y; \alpha_t x, \sigma_t^2),
$$

the finite-`t` exact annealed context can be written as

$$
p_{{\rm ctx},t}(y,u)
\propto
\exp\!\left[-\beta \left(
\frac12 k_b (u-u_{\rm eq})^2
+
\frac12 \tilde{k}_c(t) (y-\alpha_t u)^2
\right)\right],
$$

with

$$
\tilde{k}_c(t) = \frac{k_c}{\alpha_t^2 - \beta k_c \sigma_t^2}.
$$

This is chosen so that the forward kernel and the annealed context satisfy the matching condition

$$
\int dy\; q_t(y \mid x)\, p_{{\rm ctx},t}(y,u)
\propto
\exp[-\beta W(x,u)].
$$

So when the context uses the same diffusion time `t`, marginalizing out `y` reproduces the original physical coupling exactly.

In this notebook, the same construction is used with a small extension: we allow the context time to be fixed at

$$
t_{\rm ctx} = \min(t_{\rm list}),
$$

while the forward kernel time can vary across replicas. If `t_list = [t]`, this reduces automatically to the standard single-`t` case.

## 3. Critical Condition

The annealed-context formula is well-defined only when the denominator in `\tilde{k}_c(t)` is positive, i.e.

$$
\alpha_t^2 - \beta k_c \sigma_t^2 > 0.
$$

Equivalently,

$$
\bar\alpha_t > \frac{\beta k_c}{1 + \beta k_c}.
$$

This is the critical condition for the finite-`t` exact Gaussian context. In practice, this imposes an upper bound on the usable diffusion time.

<!-- Below, the notebook prints this condition for the chosen setup, then runs either:

- a single-`t` GG-PA simulation when `len(t_list) == 1`, or
- a fixed-context GG-PA replica-exchange simulation when `len(t_list) > 1`.

The signal update can be performed either by Langevin sampling or by direct Gaussian sampling through `signal_sampler='exact'`. -->

In [ ]:
import contextlib
import io
import math
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists() or not (ROOT / 'checkpoints').exists():
    raise RuntimeError('Run this notebook from the project root or from the notebooks/ directory.')
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from pymbar import timeseries as pymbar_timeseries

from ggpa.models import SimpleDiffusion
from ggpa.systems.doublewell import (
    CoupledDoubleWell,
    DoubleWellVPForwardProcess,
    finite_t_exact_context_stiffness,
    sample_1d_equilibrium,
)

seed = 7
np.random.seed(seed)
torch.manual_seed(seed)

warnings.filterwarnings('ignore', category=UserWarning, module=r'torch\.cuda')
sns.set_theme(style='whitegrid', context='notebook')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
ckpt_path = ROOT / 'checkpoints' / 'doublewell_prior.pt'
diffusion = SimpleDiffusion.load_from_checkpoint(ckpt_path, device=device)
diffusion.eval()

diffusion = torch.compile(diffusion, mode='reduce-overhead')

system = CoupledDoubleWell(a=8.0, b=16.0, c=0.0, k_b=1.0, k_c=4.0, u_eq=1.0)
forward_process = DoubleWellVPForwardProcess(diffusion.noise_scheduler)

single_t_list = [0.10] # Non-RE Sampler
re_t_list = [0.10, 0.20, 0.40, 0.80] # RE Sampler
production_t = min(re_t_list)
t_ctx = production_t
signal_sampler = 'exact'  # choose 'langevin' or 'exact'

alpha_ctx = float(forward_process.alpha(t_ctx))
sigma_ctx = float(forward_process.sigma(t_ctx))
k_ctx = float(
    finite_t_exact_context_stiffness(
        t_ctx,
        system=system,
        forward_process=forward_process,
        beta=1.0,
    )
)

print('Checkpoint loaded from:', str(ckpt_path))
print('System params:', vars(system))
print(f'single_t_list = {single_t_list}')
print(f're_t_list = {re_t_list}')
print(f'production_t = {production_t:.2f} for both non-RE and RE')
print(f't_ctx = {t_ctx:.2f}')
print(f'signal_sampler = {signal_sampler}')
print(f'context params: alpha_ctx={alpha_ctx:.4f}, sigma_ctx={sigma_ctx:.4f}, k_ctx={k_ctx:.4f}')
print('Forward-kernel parameters on the RE ladder:')
for t in re_t_list:
    alpha_t = float(forward_process.alpha(t))
    sigma_t = float(forward_process.sigma(t))
    print(f'  t={t:.2f}  alpha={alpha_t:.4f}  sigma={sigma_t:.4f}')

In [3]:
class FixedContextDoubleWellSampler:
    def __init__(
        self,
        diffusion,
        system,
        *,
        t_list,
        t_ctx=None,
        signal_sampler='langevin',
        beta=1.0,
        device='cpu',
        step_size=5e-3,
        n_signal_steps=200,
        max_step_size=0.01,
    ):
        if len(t_list) < 1:
            raise ValueError('t_list must contain at least one diffusion time.')

        self.t_list = [float(t) for t in t_list]
        if np.any(np.diff(self.t_list) < 0.0):
            raise ValueError('t_list should be sorted in ascending order.')

        self.diffusion = diffusion
        self.system = system
        self.beta = float(beta)
        self.device = torch.device(device)
        self.forward_process = DoubleWellVPForwardProcess(diffusion.noise_scheduler)
        self.t_ctx = float(min(self.t_list) if t_ctx is None else t_ctx)
        self.signal_sampler = str(signal_sampler).lower()
        if self.signal_sampler not in {'langevin', 'exact'}:
            raise ValueError("signal_sampler must be 'langevin' or 'exact'.")

        self.use_replica_exchange = len(self.t_list) > 1
        self.production_index = int(np.argmin(np.asarray(self.t_list)))
        self.step_size = float(step_size)
        self.n_signal_steps = int(n_signal_steps)
        self.max_step_size = float(max_step_size)

        self.alpha_ctx = float(self.forward_process.alpha(self.t_ctx))
        self.k_ctx = float(
            finite_t_exact_context_stiffness(
                self.t_ctx,
                system=self.system,
                forward_process=self.forward_process,
                beta=self.beta,
            )
        )

    @staticmethod
    def _as_vector(value, n=None, default=None):
        if value is None:
            if default is None:
                raise ValueError('Missing required value with no default.')
            if n is None:
                raise ValueError('Need n when broadcasting a default.')
            return np.full(int(n), float(default), dtype=np.float64)

        arr = np.asarray(value, dtype=np.float64)
        if arr.ndim == 0:
            if n is None:
                return arr.reshape(1)
            return np.full(int(n), float(arr), dtype=np.float64)

        arr = arr.reshape(-1)
        if n is None or arr.size == int(n):
            return arr.copy()
        if arr.size == 1:
            return np.full(int(n), float(arr[0]), dtype=np.float64)
        raise ValueError(f'Could not broadcast array of shape {arr.shape} to length {n}.')

    def vp_params(self, t):
        alpha = float(self.forward_process.alpha(t))
        sigma = float(self.forward_process.sigma(t))
        return alpha, sigma

    @torch.no_grad()
    def update_x(self, y, t, seed=None):
        if seed is not None:
            torch.manual_seed(int(seed))
            if self.device.type == 'cuda':
                torch.cuda.manual_seed_all(int(seed))

        y_vec = self._as_vector(y)
        y_t = torch.tensor(y_vec[:, None], dtype=torch.float32, device=self.device)
        x_clean = self.diffusion.reverse(y_t, t_start=t, t_end=0.0)
        return x_clean.detach().cpu().numpy().astype(np.float64).reshape(-1)

    def _context_force(self, y, u):
        residual = y - self.alpha_ctx * u
        force_y = -self.beta * self.k_ctx * residual
        force_u = -self.beta * self.system.k_b * (u - self.system.u_eq) + self.beta * self.alpha_ctx * self.k_ctx * residual
        return force_y, force_u

    def _total_force(self, x, y, u, t_forward):
        x = self._as_vector(x)
        y = self._as_vector(y, n=x.size)
        u = self._as_vector(u, n=x.size)

        alpha_forward, sigma_forward = self.vp_params(t_forward)
        force_y_forward = -(y - alpha_forward * x) / (sigma_forward * sigma_forward)
        force_y_ctx, force_u_ctx = self._context_force(y, u)
        return force_y_forward + force_y_ctx, force_u_ctx

    def update_signal_langevin(self, x, y, u, t_forward, *, rng=None):
        if rng is None:
            rng = np.random.default_rng()

        x = self._as_vector(x)
        y = self._as_vector(y, n=x.size)
        u = self._as_vector(u, n=x.size)

        for _ in range(self.n_signal_steps):
            force_y, force_u = self._total_force(x, y, u, t_forward)
            grad_norm = np.sqrt(force_y * force_y + force_u * force_u)
            predicted_drift = grad_norm * self.step_size
            scale = self.max_step_size / np.maximum(predicted_drift, 1e-10)
            scale = np.minimum(scale, 1.0)
            dt_actual = self.step_size * scale
            sqrt_2dt = np.sqrt(2.0 * dt_actual)

            y = y + force_y * dt_actual + sqrt_2dt * rng.standard_normal(y.shape)
            u = u + force_u * dt_actual + sqrt_2dt * rng.standard_normal(u.shape)

        return y, u

    def update_signal_exact(self, x, t_forward, *, rng=None):
        if rng is None:
            rng = np.random.default_rng()

        x = self._as_vector(x)
        alpha_forward, sigma_forward = self.vp_params(t_forward)

        if sigma_forward < 1e-12:
            y = alpha_forward * x
            precision_u = self.beta * (self.system.k_b + (self.alpha_ctx ** 2) * self.k_ctx)
            mean_u = (
                self.beta * self.system.k_b * self.system.u_eq
                + self.beta * self.alpha_ctx * self.k_ctx * y
            ) / precision_u
            std_u = math.sqrt(1.0 / precision_u)
            u = mean_u + std_u * rng.standard_normal(x.shape)
            return y, u

        inv_s2 = 1.0 / (sigma_forward * sigma_forward)
        q11 = inv_s2 + self.beta * self.k_ctx
        q12 = -self.beta * self.alpha_ctx * self.k_ctx
        q22 = self.beta * (self.system.k_b + (self.alpha_ctx ** 2) * self.k_ctx)
        det = q11 * q22 - q12 * q12

        cov = np.array([[q22, -q12], [-q12, q11]], dtype=np.float64) / det
        chol = np.linalg.cholesky(cov)

        b1 = (alpha_forward * inv_s2) * x
        b2 = self.beta * self.system.k_b * self.system.u_eq * np.ones_like(x)
        mean_y = cov[0, 0] * b1 + cov[0, 1] * b2
        mean_u = cov[1, 0] * b1 + cov[1, 1] * b2
        mean = np.stack([mean_y, mean_u], axis=1)

        z = rng.standard_normal(mean.shape)
        samples = mean + z @ chol.T
        return samples[:, 0], samples[:, 1]

    def update_signal(self, x, y, u, t_forward, *, rng=None):
        if self.signal_sampler == 'exact':
            return self.update_signal_exact(x, t_forward, rng=rng)
        return self.update_signal_langevin(x, y, u, t_forward, rng=rng)

    def _attempt_swaps_once(self, x_reps, y_reps, u_reps, *, odd, rng):
        accepted = attempted = 0
        start_idx = 1 if odd else 0
        batch_size = x_reps.shape[1]

        for i in range(start_idx, len(self.t_list) - 1, 2):
            j = i + 1
            t_i = self.t_list[i]
            t_j = self.t_list[j]

            x_i = x_reps[i][:, None]
            x_j = x_reps[j][:, None]
            y_i = y_reps[i][:, None]
            y_j = y_reps[j][:, None]

            log_i_i = np.asarray(self.forward_process.log_q_fwd(y_i, x_i, t_i), dtype=np.float64)
            log_j_j = np.asarray(self.forward_process.log_q_fwd(y_j, x_j, t_j), dtype=np.float64)
            log_i_j = np.asarray(self.forward_process.log_q_fwd(y_j, x_j, t_i), dtype=np.float64)
            log_j_i = np.asarray(self.forward_process.log_q_fwd(y_i, x_i, t_j), dtype=np.float64)

            log_alpha = (log_i_j + log_j_i) - (log_i_i + log_j_j)
            accept = (log_alpha >= 0.0) | (np.log(rng.random(batch_size)) < log_alpha)

            if np.any(accept):
                x_i_old = x_reps[i].copy()
                y_i_old = y_reps[i].copy()
                u_i_old = u_reps[i].copy()

                x_reps[i, accept] = x_reps[j, accept]
                y_reps[i, accept] = y_reps[j, accept]
                u_reps[i, accept] = u_reps[j, accept]

                x_reps[j, accept] = x_i_old[accept]
                y_reps[j, accept] = y_i_old[accept]
                u_reps[j, accept] = u_i_old[accept]

            accepted += int(np.sum(accept))
            attempted += batch_size

        return accepted, attempted

    def run(
        self,
        *,
        num_loops=500,
        update_steps=1,
        batch_size=250,
        x0=1.0,
        y0=1.0,
        u0=1.0,
        seed=0,
        record_interval=1,
        verbose=False,
    ):
        if record_interval <= 0:
            raise ValueError('record_interval must be positive.')

        rng = np.random.default_rng(seed)
        num_replicas = len(self.t_list)
        batch_size = int(batch_size)
        num_loops = int(num_loops)
        update_steps = int(update_steps)
        record_interval = int(record_interval)

        x_init = self._as_vector(x0, n=batch_size)
        y_init = self._as_vector(y0, n=batch_size)
        u_init = self._as_vector(u0, n=batch_size)

        x_reps = np.repeat(x_init[None, :], num_replicas, axis=0)
        y_reps = np.repeat(y_init[None, :], num_replicas, axis=0)
        u_reps = np.repeat(u_init[None, :], num_replicas, axis=0)

        history = {
            'outer_step': [0],
            'x': [x_reps[self.production_index].copy()],
            'y': [y_reps[self.production_index].copy()],
            'u': [u_reps[self.production_index].copy()],
        }
        swap_acc = 0
        swap_att = 0

        for loop_idx in range(num_loops):
            for _ in range(update_steps):
                for replica_idx, t in enumerate(self.t_list):
                    denoise_seed = int(rng.integers(0, 2**31 - 1))
                    x_reps[replica_idx] = self.update_x(y_reps[replica_idx], t, seed=denoise_seed)
                    y_reps[replica_idx], u_reps[replica_idx] = self.update_signal(
                        x_reps[replica_idx],
                        y_reps[replica_idx],
                        u_reps[replica_idx],
                        t,
                        rng=rng,
                    )

                if self.use_replica_exchange:
                    n_acc, n_att = self._attempt_swaps_once(
                        x_reps,
                        y_reps,
                        u_reps,
                        odd=(loop_idx % 2 == 1),
                        rng=rng,
                    )
                    swap_acc += n_acc
                    swap_att += n_att

            if (loop_idx + 1) % record_interval == 0:
                history['outer_step'].append(loop_idx + 1)
                history['x'].append(x_reps[self.production_index].copy())
                history['y'].append(y_reps[self.production_index].copy())
                history['u'].append(u_reps[self.production_index].copy())

            if verbose and (loop_idx + 1) % 100 == 0:
                print(
                    f'Loop {loop_idx + 1}/{num_loops}  '
                    f'<x_prod>={np.mean(x_reps[self.production_index]):+.4f}  '
                    f'swap_rate={swap_acc / max(swap_att, 1):.4f}'
                )

        return {
            'mode': 'replica_exchange' if self.use_replica_exchange else 'single_t',
            'signal_sampler': self.signal_sampler,
            't_list': np.asarray(self.t_list, dtype=np.float64),
            't_ctx': float(self.t_ctx),
            'outer_step': np.asarray(history['outer_step'], dtype=np.int64),
            'x_trace': np.asarray(history['x'], dtype=np.float64),
            'y_trace': np.asarray(history['y'], dtype=np.float64),
            'u_trace': np.asarray(history['u'], dtype=np.float64),
            'x_final': x_reps[self.production_index].copy(),
            'y_final': y_reps[self.production_index].copy(),
            'u_final': u_reps[self.production_index].copy(),
            'swap_rate': swap_acc / max(swap_att, 1) if self.use_replica_exchange else 0.0,
            'x_replicas': x_reps.copy(),
            'y_replicas': y_reps.copy(),
            'u_replicas': u_reps.copy(),
        }

## Run the Same `t=0.10` Production Replica With and Without RE

The next cell uses the same numerical settings in both runs:

- `250` particles
- `500` outer loops
- `200` signal-update steps per loop
- `dt = 5e-3`
- `max_step_size = 0.01`
- initialization from the right basin: `x0 = y0 = u0 = 1.0`

The only difference is the replica ladder:

- non-RE: `t_list = [0.10]`
- RE: `t_list = [0.10, 0.20, 0.40, 0.80]`

In both cases we analyze the same production time `t = 0.10`, discard the first `20%` of recorded frames as burn-in, and then compare:

- the `x` marginal via `sns.kdeplot`
- the autocorrelation of `sgn(x)` via `pymbar.timeseries`

In [ ]:
dt = 5e-3
n_signal_steps = 200
max_step_size = 0.01
num_samples = 250
num_loops = 500
update_steps = 1
record_interval = 1
burn_in = 0.2
run_seed = seed + 100


def run_configuration(t_list, label):
    sampler = FixedContextDoubleWellSampler(
        diffusion,
        system,
        t_list=t_list,
        t_ctx=t_ctx,
        signal_sampler=signal_sampler,
        device=device,
        step_size=dt,
        n_signal_steps=n_signal_steps,
        max_step_size=max_step_size,
    )

    result = sampler.run(
        num_loops=num_loops,
        update_steps=update_steps,
        batch_size=num_samples,
        x0=1.0,
        y0=1.0,
        u0=1.0,
        seed=run_seed,
        record_interval=record_interval,
        verbose=False,
    )

    x_trace = np.asarray(result['x_trace'], dtype=np.float64)
    total_frames = x_trace.shape[0]
    start_frame = int(total_frames * burn_in)
    x_samples = x_trace[start_frame:].reshape(-1)

    return {
        'label': label,
        'result': result,
        'x_trace': x_trace,
        'x_samples': x_samples,
        'start_frame': start_frame,
    }


single_run = run_configuration(single_t_list, 'GG-PA (single $t=0.10$)')
re_run = run_configuration(re_t_list, 'GG-PA-RE (production $t=0.10$)')

num_reference_samples = re_run['x_samples'].size
direct_diffusion_x = diffusion.sample(num_reference_samples, device=device).detach().cpu().numpy().reshape(-1)
reference_x = sample_1d_equilibrium(
    system.effective_potential,
    num_reference_samples,
    rng=np.random.default_rng(seed + 2000),
)

for run in [single_run, re_run]:
    result = run['result']
    print(f"{run['label']}")
    print(f"  mode = {result['mode']}")
    print(f"  t_list = {result['t_list'].tolist()}")
    print(f"  t_ctx = {result['t_ctx']:.2f}")
    print(f"  signal_sampler = {result['signal_sampler']}")
    print(f"  swap_rate = {result['swap_rate']:.4f}")
    print(f"  production trace shape = {run['x_trace'].shape}")
    print(f"  post-burn-in samples = {run['x_samples'].size}")
    print(f"  final mean(x) = {result['x_final'].mean():+.4f}")
    print(f"  final std(x) = {result['x_final'].std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.6))
sns.kdeplot(x=direct_diffusion_x, ax=ax, fill=False, bw_adjust=0.7, linewidth=2.1, color='#9ca3af', label='Direct diffusion')
sns.kdeplot(x=single_run['x_samples'], ax=ax, fill=False, bw_adjust=0.7, linewidth=2.3, color='#2563eb', label=single_run['label'])
sns.kdeplot(x=re_run['x_samples'], ax=ax, fill=False, bw_adjust=0.7, linewidth=2.3, color='#0f766e', label=re_run['label'])
sns.kdeplot(x=reference_x, ax=ax, fill=False, bw_adjust=0.7, linewidth=2.1, color='#dc2626', label='Reference')
ax.set_xlabel('x')
ax.set_ylabel('density')
ax.set_title('Coupled Double-Well: Same $t=0.10$ Production Replica, With vs Without RE')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

### Comparison: Non-RE vs. RE

As expected, the non-RE run at `t=0.10` shows poor mixing and fails to cross the barrier, while the RE run shows good mixing and correctly captures the bimodal distribution. The autocorrelation analysis further confirms that the RE run has much faster decorrelation compared to the non-RE run.

In [ ]:
def sign_trajectory_list(run):
    x_trace = np.asarray(run['x_trace'][run['start_frame']:], dtype=np.float64)
    sign_trace = np.sign(x_trace)
    sign_trace[sign_trace == 0.0] = 1.0
    return [sign_trace[:, idx].copy() for idx in range(sign_trace.shape[1])]


def compute_sign_acf_stats(sign_traces, max_lag):
    concatenated = np.concatenate(sign_traces)
    if np.isclose(np.var(concatenated), 0.0):
        acf = np.ones(max_lag, dtype=np.float64)
        return np.inf, acf, 'no sign change observed after burn-in'

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        iat = float(pymbar_timeseries.integrated_autocorrelation_timeMultiple(sign_traces, fast=False))
        acf = np.asarray(
            pymbar_timeseries.normalized_fluctuation_correlation_function_multiple(
                sign_traces,
                N_max=max_lag,
            ),
            dtype=np.float64,
        )

    if not np.isfinite(iat):
        iat = np.inf
    acf = np.nan_to_num(acf, nan=1.0, posinf=1.0, neginf=0.0)
    return iat, acf, 'PyMBAR estimate'


single_sign_traces = sign_trajectory_list(single_run)
re_sign_traces = sign_trajectory_list(re_run)

max_lag = min(100, len(single_sign_traces[0]), len(re_sign_traces[0]))
single_iat, single_acf, single_note = compute_sign_acf_stats(single_sign_traces, max_lag)
re_iat, re_acf, re_note = compute_sign_acf_stats(re_sign_traces, max_lag)

lags = np.arange(single_acf.size)
fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.plot(lags, single_acf, linewidth=2.1, color='#2563eb', label=f'{single_run["label"]}: IAT={single_iat:.2f}' if np.isfinite(single_iat) else f'{single_run["label"]}: IAT=inf')
ax.plot(lags, re_acf, linewidth=2.1, color='#0f766e', label=f'{re_run["label"]}: IAT={re_iat:.2f}' if np.isfinite(re_iat) else f'{re_run["label"]}: IAT=inf')
ax.axhline(0.0, color='black', linewidth=1.0, alpha=0.6)
ax.set_xlabel('lag')
ax.set_ylabel(r'ACF of $\mathrm{sgn}(x)$')
ax.set_title(r'PyMBAR Autocorrelation of $\mathrm{sgn}(x)$ at the Production Time $t=0.10$')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

print('Integrated autocorrelation time from PyMBAR for sgn(x):')
print(f'  {single_run["label"]}: IAT = {single_iat:.2f}' if np.isfinite(single_iat) else f'  {single_run["label"]}: IAT = inf')
print(f'    note: {single_note}')
print(f'  {re_run["label"]}: IAT = {re_iat:.2f}' if np.isfinite(re_iat) else f'  {re_run["label"]}: IAT = inf')
print(f'    note: {re_note}')